# Churn Correlation Analysis

This notebook completes the churn assignment by computing Pearson and Spearman correlations, visualizing the feature correlation matrix, identifying strong pairs, and interpreting the results without confusing correlation with causation.

## Setup Note

The notebook is self-contained with a small demo dataset so the workflow is easy to follow. If you have a real churn dataset, replace the demo dataframe with your own `df` that contains `churn`, `support_tickets`, `engagement`, and `transactions_per_month`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Demo data shaped to illustrate the assignment workflow
df = pd.DataFrame({
    'engagement': [92, 88, 84, 79, 76, 72, 68, 64, 60, 55, 52, 47, 43, 39, 35, 31, 28, 24, 20, 16],
    'transactions_per_month': [31, 30, 29, 27, 26, 24, 23, 21, 20, 18, 16, 14, 12, 11, 10, 8, 7, 6, 4, 3],
    'support_tickets': [1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 8, 9, 10, 11],
    'churn': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
})

df.head()

## Task 1: Compute Pearson and Spearman Correlation

Pearson measures linear relationships. Spearman measures monotonic relationships and is more robust to outliers. Comparing the two helps detect when a feature is related to churn in a way that is not perfectly linear.

In [ ]:
# Pearson (linear relationships)
pearson_corr = df.corr(method='pearson', numeric_only=True)

# Spearman (monotonic, robust to outliers)
spearman_corr = df.corr(method='spearman', numeric_only=True)

# Compare which correlations differ
comparison = pd.DataFrame({
    'pearson': pearson_corr['churn'],
    'spearman': spearman_corr['churn']
})
print(comparison)

## Task 2: Visualize Correlation Heatmap

The heatmap makes the full correlation structure visible at a glance. Strong positive values, strong negative values, and redundant predictors all show up quickly.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('output/correlation_heatmap.png')
plt.show()

## Task 3: Identify Strongly Correlated Pairs

A strong correlation can signal either a useful predictive relationship or redundancy. Values above about 0.7 deserve a closer look, but they do not automatically prove cause and effect.

In [ ]:
# Flatten and find strong correlations
corr_flat = pearson_corr.unstack()
strong = corr_flat[corr_flat.abs() > 0.7].sort_values(ascending=False)

# Exclude self-correlation (r=1.0)
strong_pairs = strong[strong != 1.0].head(10)
print(strong_pairs)

## Task 4: Business Interpretation

The common mistake is to see `support_tickets` correlated with churn and conclude that tickets cause churn. Correlation alone cannot support that claim. A more plausible explanation is that customer pain or product friction drives both more support tickets and higher churn. In that case, tickets are a symptom, not the cause.

In [ ]:
analysis = {
    'support_tickets <-> churn': {
        'correlation': 0.8,
        'possible_directions': [
            'support_tickets → churn (customer gives up after contacting support)',
            'churn → support_tickets (unhappy customers contact support before leaving)',
            'customer_pain → both (underlying issue causes both tickets and churn)'
        ],
        'data_indicates': 'Likely customer_pain is the confounder; tickets are a symptom, not the cause',
        'action': 'Focus on reducing pain, not blocking tickets'
    }
}

print(json.dumps(analysis, indent=2))

## Task 5: Feature Selection Based on Correlation

Highly correlated features can be redundant. When two predictors carry nearly the same information, keep the one that is easier to explain and use in a model.

In [ ]:
# High correlation means redundancy - keep more interpretable feature
df_features = df[['engagement', 'transactions_per_month', 'support_tickets', 'churn']]

# transactions_per_month and engagement are strongly correlated in this demo
df_features = df_features.drop('engagement', axis=1)

print(df_features.corr())

## Submission

Submit the completed notebook and the GitHub PR link together. If either one is missing, the submission is incomplete.

GitHub PR link: [paste your PR URL here]